# 🌎 GeoIA Colombia · TerraDrop → KPI → Prompt para Lovable

Este cuaderno toma las **capas que descarga TerraDrop** (o las consulta en vivo a los servicios oficiales) para un **municipio o una vereda**, calcula **KPI e indicadores** y termina entregándote:

| Archivo | Para qué sirve |
|---|---|
| `prompt_lovable.md` | El prompt completo (en español) para crear el dashboard en [Lovable](https://lovable.dev) |
| `prompt_lovable_corto.md` | Versión corta para el primer mensaje; luego iteras con el completo |
| `kpis.json` | Todos los indicadores, listos para alimentar el dashboard |
| `geojson/*.geojson` | Territorio y capas **livianas** (simplificadas) para el mapa interactivo |
| `tablas/*.csv` | Tablas de cobertura, áreas protegidas, hidrografía, vías y veredas |
| `mapa_ubicacion.png` | Mapa de ubicación (Colombia → departamento → territorio con capas) |

**Grupos de KPI:** 1) Territorio y cobertura · 2) Áreas protegidas · 3) Hidrografía · 4) Vías y accesibilidad (+ cruces entre capas y tabla por vereda).

### Cómo usarlo
1. Ejecuta las celdas en orden (**Entorno → Entorno de ejecución → Ejecutar todo**).
2. En la celda *Parámetros* elige el origen:
   * **auto** (recomendado): usa el ZIP de TerraDrop si lo subes; si no, consulta las APIs con el código DANE.
   * **zip**: solo el ZIP que descargaste en TerraDrop (`terradrop_<código>_<municipio>.zip`).
   * **api**: consulta en vivo con `CODIGO_MUNICIPIO` (5 dígitos) y, si quieres una vereda, `CODIGO_VEREDA` (8 dígitos).
3. Al final se descarga `paquete_lovable_….zip`. Sube su contenido a Lovable junto con `prompt_lovable.md`.

> Las áreas y longitudes se calculan en **MAGNA-SIRGAS Origen Nacional (EPSG:9377)**. Los datos son los de cada fuente oficial; revisa su vigencia en la página *Fuentes y calidad* de TerraDrop.

In [ ]:
#@title 1 · Preparar el entorno { display-mode: "form" }
import importlib, subprocess, sys

def _asegurar(paquete, modulo):
    try:
        importlib.import_module(modulo)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

for _p, _m in [("geopandas", "geopandas"), ("pyogrio", "pyogrio"), ("shapely>=2.0", "shapely"),
               ("pyproj", "pyproj"), ("matplotlib", "matplotlib"), ("requests", "requests")]:
    _asegurar(_p, _m)

import os, re, io, json, math, time, zipfile, shutil, datetime, warnings, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import shapely
from shapely.geometry import box, Point
from shapely.geometry.polygon import orient
from shapely.ops import unary_union

warnings.filterwarnings("ignore")
print("Listo · geopandas", gpd.__version__, "· shapely", shapely.__version__)

In [ ]:
#@title 2 · Parámetros { display-mode: "form" }
ORIGEN = "auto"            #@param ["auto", "zip", "api"]
RUTA_ZIP = ""              #@param {type:"string"}
CODIGO_MUNICIPIO = ""      #@param {type:"string"}
CODIGO_VEREDA = ""         #@param {type:"string"}
NOMBRE_PROYECTO = ""       #@param {type:"string"}
CARPETA_SALIDA = "salida_terradrop"  #@param {type:"string"}
DISTANCIAS_ACCESO_KM = [1, 3]        # franjas de accesibilidad alrededor de las vías

# Cómo se decide la fuente de datos cuando ORIGEN = "auto":
#   1) si RUTA_ZIP existe -> se usa ese ZIP;
#   2) si estás en Colab -> se te ofrece subir el ZIP de TerraDrop (puedes cancelar);
#   3) si hay CODIGO_MUNICIPIO -> se consultan las APIs oficiales.

In [ ]:
#@title 3 · Catálogo de capas y utilidades { display-mode: "form" }
IDEAM = "https://visualizador.ideam.gov.co/gisserver/rest/services"
INTERES = "https://services2.arcgis.com/RVvWzU3lgJISqdke/arcgis/rest/services/Interes_Colombia/FeatureServer"

# Mismo catálogo que usa TerraDrop (descargas.html). spatial=True -> se filtra con el polígono del territorio.
SERV = {
    "municipio":           dict(nombre="Límite municipal", kind="poly", page=1000, spatial=False,
                                fuente="DANE · Marco Geoestadístico Nacional (vía IDEAM)", url=IDEAM + "/Entidades_Territoriales/FeatureServer/2/query"),
    "veredas":             dict(nombre="Veredas DANE", kind="poly", page=1000, spatial=False,
                                fuente="DANE · Veredas (vía IDEAM)", url=IDEAM + "/Entidades_Territoriales/FeatureServer/5/query"),
    "runap":               dict(nombre="Áreas protegidas (RUNAP)", kind="poly", page=500, spatial=True,
                                fuente="MinAmbiente · RUNAP (OTL Área protegida)", url="https://geo.minambiente.gov.co/server/rest/services/OTL/OTL_Area_Protegida/FeatureServer/0/query"),
    "vias_pavimentadas":   dict(nombre="Vías pavimentadas", kind="line", page=1000, spatial=True,
                                fuente="IGAC · Interés Colombia", url=INTERES + "/16/query"),
    "vias_sin_pavimentar": dict(nombre="Vías sin pavimentar", kind="line", page=1000, spatial=True,
                                fuente="IGAC · Interés Colombia", url=INTERES + "/15/query"),
    "vias_tiempo_seco":    dict(nombre="Vías transitables en tiempo seco", kind="line", page=1000, spatial=True,
                                fuente="IGAC · Interés Colombia", url=INTERES + "/14/query"),
    "drenajes":            dict(nombre="Drenajes (hidrografía)", kind="line", page=1000, spatial=True,
                                fuente="Capa HIDROGRAFIA · ArcGIS Online", url="https://services.arcgis.com/vC1CdlKWEAtuT38d/ArcGIS/rest/services/HIDROGRAFIA/FeatureServer/0/query"),
    "rios_principales":    dict(nombre="Ríos principales", kind="line", page=1000, spatial=True,
                                fuente="IGAC · Interés Colombia (no es la red completa)", url=INTERES + "/8/query"),
    "clc2024":             dict(nombre="Coberturas de la tierra CLC 2024", kind="poly", page=1000, spatial=True,
                                fuente="IDEAM · Corine Land Cover 1:100.000", url=IDEAM + "/Estado_Cobertura_Tierra/MapServer/10/query"),
}
URL_DEPARTAMENTOS = IDEAM + "/Entidades_Territoriales/FeatureServer/4/query"
CAPAS = list(SERV)
VIAS = {"vias_pavimentadas": "Pavimentadas", "vias_sin_pavimentar": "Sin pavimentar", "vias_tiempo_seco": "Transitables solo en tiempo seco"}

CRS_METRICO = "EPSG:9377"   # MAGNA-SIRGAS / Origen-Nacional
try:
    import pyproj
    pyproj.CRS.from_user_input(CRS_METRICO)
except Exception:
    CRS_METRICO = "EPSG:3116"  # respaldo (MAGNA-SIRGAS / Colombia Bogotá)

# Corine Land Cover Colombia: niveles 1 y 2 (el nivel 3 viene en la columna "leyenda" de la capa)
CLC_N1 = {"1": "Territorios artificializados", "2": "Territorios agrícolas", "3": "Bosques y áreas seminaturales",
          "4": "Áreas húmedas", "5": "Superficies de agua"}
CLC_N2 = {"11": "Zonas urbanizadas", "12": "Zonas industriales o comerciales y redes de comunicación",
          "13": "Zonas de extracción minera e hidrocarburos y escombreras", "14": "Zonas verdes artificializadas no agrícolas",
          "21": "Cultivos transitorios", "22": "Cultivos permanentes", "23": "Pastos", "24": "Áreas agrícolas heterogéneas",
          "31": "Bosques", "32": "Vegetación herbácea y/o arbustiva", "33": "Áreas abiertas, sin o con poca vegetación",
          "41": "Áreas húmedas continentales", "42": "Áreas húmedas costeras", "51": "Aguas continentales", "52": "Aguas marítimas"}
CLC_COL = {"1": "#d95a5a", "2": "#e8c34b", "3": "#3fa56b", "4": "#7fc4b8", "5": "#4d8fe8"}   # mismos colores de TerraDrop
NATURALES = ("3", "4", "5")   # cobertura natural y seminatural = bosques/seminaturales + húmedas + agua

# Dominios codificados del RUNAP (servicio de MinAmbiente)
RUNAP_CAT = {3: ("Público", "Vía Parque", True), 4: ("Público", "Parque Natural Regional", False),
             5: ("Público", "Área Natural Única", True), 6: ("Público", "Distrito de Manejo Integrado", False),
             7: ("Privado", "Reserva Natural de la Sociedad Civil", False), 8: ("Público", "Reserva Forestal Protectora", False),
             9: ("Público", "Santuario de Flora", True), 10: ("Público", "Reserva Natural", True),
             11: ("Público", "Distrito de Conservación de Suelos", False), 12: ("Público", "Área de Recreación", False),
             13: ("Público", "Santuario de Flora y Fauna", True), 14: ("Público", "Parque Nacional Natural", True),
             15: ("Público", "Santuario de Fauna", True)}
RUNAP_ESTADO = {42: "Inscrita", 43: "Registrada"}
RUNAP_AMBITO = {63: "Nacional", 64: "Local", 65: "Regional"}
DEPARTAMENTOS = {"05": "Antioquia", "08": "Atlántico", "11": "Bogotá D.C.", "13": "Bolívar", "15": "Boyacá", "17": "Caldas",
                 "18": "Caquetá", "19": "Cauca", "20": "Cesar", "23": "Córdoba", "25": "Cundinamarca", "27": "Chocó",
                 "41": "Huila", "44": "La Guajira", "47": "Magdalena", "50": "Meta", "52": "Nariño", "54": "Norte de Santander",
                 "63": "Quindío", "66": "Risaralda", "68": "Santander", "70": "Sucre", "73": "Tolima", "76": "Valle del Cauca",
                 "81": "Arauca", "85": "Casanare", "86": "Putumayo", "88": "San Andrés, Providencia y Santa Catalina",
                 "91": "Amazonas", "94": "Guainía", "95": "Guaviare", "97": "Vaupés", "99": "Vichada"}

# ---------- utilidades ----------
def col(gdf, *nombres):
    """Devuelve el nombre real de la primera columna que exista (sin distinguir mayúsculas)."""
    bajo = {c.lower(): c for c in gdf.columns}
    for n in nombres:
        if n.lower() in bajo:
            return bajo[n.lower()]
    return None

def vacio(crs="EPSG:4326"):
    return gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs=crs)

def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")

def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", sin_tildes(s).lower()).strip("_") or "territorio"

def titulo(s):
    s = str(s or "").strip()
    peq = {"de", "del", "la", "las", "el", "los", "y", "en"}
    return " ".join(w.lower() if (i and w.lower() in peq) else w.capitalize() for i, w in enumerate(s.lower().split()))

def num(x):
    """None si no es un número finito."""
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return None if (math.isnan(x) or math.isinf(x)) else x

def rd(x, n=1):
    x = num(x)
    return None if x is None else round(x, n)

def fmt(x, dec=0):
    """Formato colombiano: 1.234,5"""
    x = num(x)
    if x is None:
        return "—"
    s = f"{x:,.{dec}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def limpiar(gdf, tipo):
    """Repara geometrías y deja solo polígonos (tipo='poly') o líneas (tipo='line')."""
    if gdf is None or len(gdf) == 0:
        return vacio(gdf.crs if gdf is not None and gdf.crs else "EPSG:4326")
    g = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    if len(g) == 0:
        return vacio(gdf.crs)
    g["geometry"] = g.geometry.make_valid()
    g = g.explode(index_parts=False)
    g = g[g.geom_type == ("Polygon" if tipo == "poly" else "LineString")]
    return g[~g.geometry.is_empty].reset_index(drop=True)

def texto_col(gdf, nombre):
    c = col(gdf, nombre)
    return gdf[c].astype(str).str.strip() if c else pd.Series([""] * len(gdf), index=gdf.index)

print("Catálogo listo · CRS métrico:", CRS_METRICO, "·", len(SERV), "capas")

In [ ]:
#@title 4 · Cargar las capas (ZIP de TerraDrop o APIs) { display-mode: "form" }
# ---------------- consulta a los servicios ArcGIS REST ----------------
def _post(url, datos, intentos=3, espera=60):
    ultimo = None
    for i in range(intentos):
        try:
            r = requests.post(url, data=datos, timeout=espera)
            r.raise_for_status()
            j = r.json()
            if "error" in j:
                raise RuntimeError(j["error"].get("message", "El servicio devolvió un error"))
            return j
        except Exception as e:      # reintento con espera creciente
            ultimo = e
            time.sleep(1.5 * (i + 1))
    raise ultimo

def _anillos(geom, tol0=0.0003, max_vertices=4000):
    """Polígono simplificado -> anillos Esri (exterior en sentido horario). Pocos vértices = consulta ligera."""
    tol, anillos = tol0, []
    for _ in range(7):
        g = geom.simplify(tol, preserve_topology=True)
        polis = [g] if g.geom_type == "Polygon" else [p for p in getattr(g, "geoms", []) if p.geom_type == "Polygon"]
        anillos, n = [], 0
        for p in polis:
            if p.is_empty:
                continue
            p = orient(p, sign=-1.0)
            for r in [p.exterior, *p.interiors]:
                c = [list(xy) for xy in r.coords]
                anillos.append(c)
                n += len(c)
        if anillos and n <= max_vertices:
            break
        tol *= 2.2
    return anillos

def _filtro_esri(geom, modo):
    if modo == "envolvente":
        x0, y0, x1, y1 = geom.bounds
        return "esriGeometryEnvelope", {"xmin": x0, "ymin": y0, "xmax": x1, "ymax": y1, "spatialReference": {"wkid": 4326}}
    return "esriGeometryPolygon", {"rings": _anillos(geom), "spatialReference": {"wkid": 4326}}

def _a_gdf(feats):
    if not feats:
        return vacio()
    return gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")

def consultar(capa, where="1=1", geom=None, url=None, page=None, spatial=None, extra=None):
    """Descarga una capa completa (paginada). geom = polígono shapely en EPSG:4326 para el filtro espacial."""
    s = SERV.get(capa, {})
    url = url or s["url"]
    page = page or s.get("page", 1000)
    spatial = s.get("spatial", False) if spatial is None else spatial
    feats, offset, modo = [], 0, "poligono"
    while True:
        datos = {"where": where, "outFields": "*", "outSR": 4326, "returnGeometry": "true", "f": "geojson",
                 "resultOffset": offset, "resultRecordCount": page}
        datos.update(extra or {})
        if spatial and geom is not None:
            tipo, esri = _filtro_esri(geom, modo)
            datos.update(geometry=json.dumps(esri), geometryType=tipo, inSR=4326, spatialRel="esriSpatialRelIntersects")
        try:
            j = _post(url, datos)
        except Exception:
            if spatial and geom is not None and modo == "poligono" and offset == 0:
                modo = "envolvente"      # el servicio no aceptó el polígono: se usa la envolvente
                continue
            raise
        got = j.get("features", [])
        feats += got
        mas = j.get("exceededTransferLimit") or (j.get("properties") or {}).get("exceededTransferLimit")
        if not got or (not mas and len(got) < page):
            break
        offset += len(got)
        if offset > 250000:
            break
    return _a_gdf(feats)

def cargar_api(cod_mun, cod_ver=""):
    cod_mun = str(cod_mun).strip().zfill(5)
    capas = {"municipio": consultar("municipio", f"num_dane='{cod_mun}'")}
    if len(capas["municipio"]) == 0:
        raise ValueError(f"No se encontró el municipio {cod_mun}. Revisa el código DANE (5 dígitos).")
    donde = f"cod_ver='{cod_ver}'" if cod_ver else f"cod_mpio='{cod_mun}'"
    capas["veredas"] = consultar("veredas", donde)
    if cod_ver and len(capas["veredas"]) == 0:
        raise ValueError(f"No se encontró la vereda {cod_ver}. Revisa el código (8 dígitos).")
    base = capas["veredas"] if cod_ver else capas["municipio"]
    geom = unary_union(base.geometry.make_valid())
    for c in CAPAS:
        if c in capas:
            continue
        print(f"  · {SERV[c]['nombre']} …", end=" ")
        capas[c] = consultar(c, geom=geom)
        print(len(capas[c]), "elementos")
    return capas, {"origen": "api", "cod_mun": cod_mun, "cod_ver": cod_ver}

# ---------------- ZIP de TerraDrop ----------------
def cargar_zip(ruta):
    z = zipfile.ZipFile(ruta)
    capas, cod_mun, cod_ver = {}, "", ""
    for n in z.namelist():
        b = os.path.basename(n)
        if not b.lower().endswith(".geojson") or b.lower().startswith("asignacion_"):
            continue
        for cid in sorted(CAPAS, key=len, reverse=True):
            if b.endswith(f"_{cid}.geojson"):
                m = re.match(r"^(\d{5})_", b)
                cod_mun = cod_mun or (m.group(1) if m else "")
                v = re.search(r"_v(\d{6,8})_", b)
                cod_ver = cod_ver or (v.group(1) if v else "")
                try:
                    capas[cid] = gpd.read_file(io.BytesIO(z.read(n)))
                except Exception:
                    capas[cid] = vacio()
                if capas[cid].crs is None:
                    capas[cid] = capas[cid].set_crs("EPSG:4326")
                capas[cid] = capas[cid].to_crs("EPSG:4326")
                break
    if "municipio" not in capas:
        raise ValueError("El ZIP no trae la capa de límite municipal (…_municipio.geojson). Descárgalo de nuevo con esa capa marcada.")
    for c in CAPAS:
        capas.setdefault(c, vacio())
    return capas, {"origen": "zip", "cod_mun": cod_mun, "cod_ver": cod_ver, "zip": os.path.basename(ruta)}

# ---------------- decidir la fuente ----------------
def _subir_zip():
    try:
        from google.colab import files
    except ImportError:
        return None
    print("Sube el ZIP de TerraDrop (o cancela para consultar las APIs con el código DANE)…")
    try:
        subido = files.upload()
    except Exception:
        return None
    for nombre in subido:
        if nombre.lower().endswith(".zip"):
            return nombre
    return None

ruta = RUTA_ZIP.strip() if RUTA_ZIP else ""
if ORIGEN in ("auto", "zip") and not (ruta and os.path.exists(ruta)):
    ruta = _subir_zip() or ""
if ORIGEN in ("auto", "zip") and ruta and os.path.exists(ruta):
    print("Leyendo ZIP:", ruta)
    CAPAS_RAW, FUENTE = cargar_zip(ruta)
    if CODIGO_MUNICIPIO.strip():
        FUENTE["cod_mun"] = CODIGO_MUNICIPIO.strip().zfill(5)
    if CODIGO_VEREDA.strip():
        FUENTE["cod_ver"] = CODIGO_VEREDA.strip()
elif ORIGEN in ("auto", "api") and CODIGO_MUNICIPIO.strip():
    print("Consultando las APIs oficiales para", CODIGO_MUNICIPIO.strip(), CODIGO_VEREDA.strip() or "(todo el municipio)")
    CAPAS_RAW, FUENTE = cargar_api(CODIGO_MUNICIPIO, CODIGO_VEREDA.strip())
else:
    raise SystemExit("No hay datos de entrada: sube el ZIP de TerraDrop o escribe CODIGO_MUNICIPIO (5 dígitos) en los parámetros y vuelve a ejecutar.")

print("\nCapas cargadas:")
for c in CAPAS:
    print(f"  {SERV[c]['nombre']:<38} {len(CAPAS_RAW[c]):>6} elementos")

In [ ]:
#@title 5 · Preparar el territorio (proyección, reparación y recorte) { display-mode: "form" }
COD_MUN = FUENTE.get("cod_mun") or ""
COD_VER = FUENTE.get("cod_ver") or ""

# --- reparar y proyectar a metros ---
RAW_M = {}
for c in CAPAS:
    g = limpiar(CAPAS_RAW[c], SERV[c]["kind"])
    RAW_M[c] = g.to_crs(CRS_METRICO) if len(g) else vacio(CRS_METRICO)

mun = RAW_M["municipio"]
if len(mun) == 0:
    raise SystemExit("La capa de límite municipal está vacía; no se puede continuar.")
mun_row = CAPAS_RAW["municipio"].iloc[0]
if not COD_MUN:
    COD_MUN = str(mun_row.get(col(CAPAS_RAW["municipio"], "num_dane"), "")).zfill(5)

# --- nombres ---
c_dpto = col(CAPAS_RAW["municipio"], "nom_dpto")
dpto_val = str(mun_row.get(c_dpto, "")).strip() if c_dpto else ""
if dpto_val.isdigit():
    dpto_val = DEPARTAMENTOS.get(dpto_val.zfill(2), dpto_val)
if not dpto_val or dpto_val.lower() == "none":
    dpto_val = DEPARTAMENTOS.get(COD_MUN[:2], "")
NOM_DPTO = titulo(dpto_val) if dpto_val.isupper() or dpto_val.islower() else dpto_val
NOM_MUN = titulo(str(mun_row.get(col(CAPAS_RAW["municipio"], "nom_mpio"), COD_MUN)))

# --- territorio de estudio: vereda (si hay) o municipio ---
ver_all = RAW_M["veredas"]
c_cv, c_nv = col(ver_all, "cod_ver"), col(ver_all, "nom_ver")
if COD_VER and len(ver_all) and c_cv:
    sel = ver_all[ver_all[c_cv].astype(str) == str(COD_VER)]
    TERR = sel if len(sel) else mun
    ES_VEREDA = len(sel) > 0
else:
    TERR, ES_VEREDA = mun, False
if ES_VEREDA:
    NOM_VER = titulo(str(TERR.iloc[0][c_nv])) if c_nv else COD_VER
    NOM_TERR = f"Vereda {NOM_VER} · {NOM_MUN}"
else:
    NOM_VER = ""
    NOM_TERR = NOM_MUN
if NOMBRE_PROYECTO.strip():
    NOM_TERR_TITULO = NOMBRE_PROYECTO.strip()
else:
    NOM_TERR_TITULO = NOM_TERR
TERR_GEOM = unary_union(TERR.geometry)
MUN_GEOM = unary_union(mun.geometry)
AREA_T_HA = TERR_GEOM.area / 1e4
AREA_T_KM2 = AREA_T_HA / 100

# --- recortar todas las capas al territorio ---
D = {}
for c in CAPAS:
    g = RAW_M[c]
    if c in ("municipio", "veredas"):
        D[c] = g
        continue
    if len(g) == 0:
        D[c] = g
        continue
    try:
        r_ = gpd.clip(g, TERR_GEOM, keep_geom_type=True)
    except Exception:
        r_ = g[g.intersects(TERR_GEOM)].copy()
        r_["geometry"] = r_.geometry.intersection(TERR_GEOM)
    D[c] = limpiar(r_, SERV[c]["kind"])

# --- veredas que se tabulan: la elegida, o todas las del municipio ---
VER_M = TERR if ES_VEREDA else ver_all
if len(VER_M) and c_cv:
    VER_M = VER_M.rename(columns={c_cv: "cod_ver"}).copy()
    VER_M["nom_ver"] = VER_M[c_nv].astype(str).map(titulo) if c_nv else VER_M["cod_ver"]
else:
    VER_M = vacio(CRS_METRICO)

print(f"Territorio: {NOM_TERR_TITULO}  ({'vereda ' + COD_VER if ES_VEREDA else 'municipio ' + COD_MUN})  · {NOM_DPTO}")
print(f"Área calculada: {fmt(AREA_T_HA, 1)} ha  ·  {fmt(AREA_T_KM2, 2)} km²  ·  veredas a tabular: {len(VER_M)}")
for c in CAPAS[2:]:
    print(f"  {SERV[c]['nombre']:<38} {len(D[c]):>6} fragmentos dentro del territorio")

In [ ]:
#@title 6 · KPI · Territorio y cobertura (CLC 2024) { display-mode: "form" }
K = {}
cent = gpd.GeoSeries([TERR_GEOM], crs=CRS_METRICO).to_crs(4326).iloc[0].centroid
off_ha = None
if ES_VEREDA:
    c_a = col(TERR, "area_ha")
    off_ha = num(TERR.iloc[0][c_a]) if c_a else None

K["territorio"] = {
    "nombre": NOM_TERR_TITULO, "tipo": "vereda" if ES_VEREDA else "municipio",
    "departamento": NOM_DPTO, "municipio": NOM_MUN, "vereda": NOM_VER or None,
    "codigo_dane_municipio": COD_MUN, "codigo_vereda": COD_VER or None,
    "area_ha": rd(AREA_T_HA, 1), "area_km2": rd(AREA_T_KM2, 2),
    "area_oficial_ha": rd(off_ha, 1) if off_ha else None,
    "perimetro_km": rd(TERR_GEOM.length / 1000, 1),
    "area_municipio_ha": rd(MUN_GEOM.area / 1e4, 1),
    "porcentaje_del_municipio": rd(AREA_T_HA / (MUN_GEOM.area / 1e4) * 100, 1) if ES_VEREDA else 100.0,
    "numero_veredas_municipio": int(len(ver_all)) if not ES_VEREDA else None,
    "centroide": {"lon": round(cent.x, 5), "lat": round(cent.y, 5)},
    "crs_metrico": CRS_METRICO,
}

# ---------------- cobertura ----------------
clc = D["clc2024"].copy()
CLC_OK = len(clc) > 0 and col(clc, "codigo") is not None
if CLC_OK:
    clc["codigo"] = texto_col(clc, "codigo").str.replace(r"\.0$", "", regex=True)
    clc["n1"], clc["n2"] = clc["codigo"].str[:1], clc["codigo"].str[:2]
    c_ley = col(clc, "leyenda")
    clc["leyenda"] = clc[c_ley].astype(str).str.strip() if c_ley else clc["codigo"]
    clc["ha"] = clc.geometry.area / 1e4
    TOTAL_CLC = clc["ha"].sum()

    n1 = clc.groupby("n1")["ha"].sum().reindex(list(CLC_N1)).fillna(0)
    nivel1 = [{"codigo": k, "nombre": CLC_N1[k], "ha": rd(v, 1), "porcentaje": rd(v / TOTAL_CLC * 100, 1), "color": CLC_COL[k]}
              for k, v in n1.items()]
    n2 = clc.groupby("n2")["ha"].sum().sort_values(ascending=False)
    nivel2 = [{"codigo": k, "nombre": CLC_N2.get(k, "Otra cobertura"), "ha": rd(v, 1), "porcentaje": rd(v / TOTAL_CLC * 100, 1)}
              for k, v in n2.items() if v > 0]
    g3 = clc.groupby("codigo").agg(ha=("ha", "sum"), parches=("ha", "size"), leyenda=("leyenda", "first")).sort_values("ha", ascending=False)
    nivel3 = [{"codigo": k, "nombre": r.leyenda, "ha": rd(r.ha, 1), "porcentaje": rd(r.ha / TOTAL_CLC * 100, 1),
               "parches": int(r.parches), "clase_nivel1": k[:1]} for k, r in g3.iterrows()]
    p = g3["ha"] / TOTAL_CLC
    p = p[p > 0]
    shannon = float(-(p * np.log(p)).sum())
    equit = shannon / math.log(len(p)) if len(p) > 1 else 0.0
    nat = clc[clc.n1.isin(NATURALES)]["ha"].sum()
    trans = clc[clc.n1.isin(("1", "2"))]["ha"].sum()
    bosque = clc[clc.n2 == "31"]
    ha_bosque = bosque["ha"].sum()
    if len(bosque):
        parches = list(unary_union(bosque.geometry).geoms) if unary_union(bosque.geometry).geom_type == "MultiPolygon" else [unary_union(bosque.geometry)]
        areas_p = sorted([q.area / 1e4 for q in parches], reverse=True)
        frag = {"parches": len(areas_p), "tamano_medio_ha": rd(float(np.mean(areas_p)), 1),
                "parche_mayor_ha": rd(areas_p[0], 1), "porcentaje_en_parche_mayor": rd(areas_p[0] / ha_bosque * 100, 1)}
    else:
        frag = {"parches": 0, "tamano_medio_ha": None, "parche_mayor_ha": None, "porcentaje_en_parche_mayor": None}
    dom = nivel3[0]
    pastos = clc[clc.n2 == "23"]["ha"].sum()
    cultivos = clc[clc.n2.isin(("21", "22"))]["ha"].sum()
    K["cobertura"] = {
        "anio": 2024, "escala": "1:100.000", "area_clasificada_ha": rd(TOTAL_CLC, 1),
        "porcentaje_territorio_con_dato": rd(TOTAL_CLC / AREA_T_HA * 100, 1),
        "nivel1": nivel1, "nivel2": nivel2, "nivel3": nivel3,
        "natural_seminatural_ha": rd(nat, 1), "natural_seminatural_pct": rd(nat / TOTAL_CLC * 100, 1),
        "transformada_ha": rd(trans, 1), "transformada_pct": rd(trans / TOTAL_CLC * 100, 1),
        "bosque_ha": rd(ha_bosque, 1), "bosque_pct": rd(ha_bosque / TOTAL_CLC * 100, 1), "bosque_fragmentacion": frag,
        "pastos_ha": rd(pastos, 1), "pastos_pct": rd(pastos / TOTAL_CLC * 100, 1),
        "cultivos_ha": rd(cultivos, 1), "cultivos_pct": rd(cultivos / TOTAL_CLC * 100, 1),
        "cobertura_dominante": {"codigo": dom["codigo"], "nombre": dom["nombre"], "porcentaje": dom["porcentaje"]},
        "numero_coberturas_nivel3": len(nivel3), "shannon": rd(shannon, 3), "equitatividad": rd(equit, 3),
    }
    print(f"Cobertura dominante: {dom['nombre']} ({fmt(dom['porcentaje'],1)} %) · natural/seminatural {fmt(nat/TOTAL_CLC*100,1)} % · bosque {fmt(ha_bosque,1)} ha")
else:
    TOTAL_CLC = 0
    K["cobertura"] = None
    print("⚠ Sin datos de coberturas CLC en el territorio: esa sección quedará vacía en el dashboard.")

In [ ]:
#@title 7 · KPI · Áreas protegidas (RUNAP) { display-mode: "form" }
def _dec(v, dic, defecto="Sin dato"):
    try:
        return dic.get(int(float(v)), defecto)
    except (TypeError, ValueError):
        t = str(v).strip()
        return t if t and t.lower() != "none" else defecto

ap = D["runap"].copy()
AP_GEOM = None
if len(ap):
    c_nom, c_cat = col(ap, "uab_nombre_area_protegida", "nombre"), col(ap, "uab_categoria")
    c_est, c_amb = col(ap, "uab_estado"), col(ap, "uab_ambito_gestion")
    c_tot = col(ap, "area_total_ha")
    ap["nombre"] = ap[c_nom].astype(str).str.strip().map(titulo) if c_nom else "Área protegida"
    def _cat(v):
        try:
            return RUNAP_CAT[int(float(v))]
        except (KeyError, TypeError, ValueError):
            return ("Sin dato", str(v), False)
    cats = ap[c_cat].map(_cat) if c_cat else pd.Series([("Sin dato", "Sin dato", False)] * len(ap), index=ap.index)
    ap["naturaleza"] = cats.map(lambda t: t[0]); ap["categoria"] = cats.map(lambda t: t[1]); ap["spnn"] = cats.map(lambda t: t[2])
    ap["estado"] = ap[c_est].map(lambda v: _dec(v, RUNAP_ESTADO)) if c_est else "Sin dato"
    ap["ambito"] = ap[c_amb].map(lambda v: _dec(v, RUNAP_AMBITO)) if c_amb else "Sin dato"
    ap["total_ha"] = pd.to_numeric(ap[c_tot], errors="coerce") if c_tot else np.nan
    AP_GEOM = unary_union(ap.geometry)
    ap_ha = AP_GEOM.area / 1e4
    por_area = []
    for (nombre, categoria), g in ap.groupby(["nombre", "categoria"]):
        gm = unary_union(g.geometry)
        ha = gm.area / 1e4
        tot = num(g["total_ha"].max())
        por_area.append({"nombre": nombre, "categoria": categoria, "naturaleza": g["naturaleza"].iloc[0], "ambito": g["ambito"].iloc[0],
                         "estado": g["estado"].iloc[0], "sistema_parques_nacionales": bool(g["spnn"].iloc[0]),
                         "ha_en_territorio": rd(ha, 1), "porcentaje_del_territorio": rd(ha / AREA_T_HA * 100, 2),
                         "area_total_ha": rd(tot, 1), "porcentaje_del_area_en_territorio": rd(ha / tot * 100, 1) if tot else None})
    por_area.sort(key=lambda d: -(d["ha_en_territorio"] or 0))
    def _agr(campo):
        out = []
        for k, g in ap.groupby(campo):
            ha = unary_union(g.geometry).area / 1e4
            out.append({"nombre": k, "ha": rd(ha, 1), "porcentaje_del_territorio": rd(ha / AREA_T_HA * 100, 2),
                        "numero_areas": int(g["nombre"].nunique())})
        return sorted(out, key=lambda d: -(d["ha"] or 0))
    K["areas_protegidas"] = {
        "hay_traslape": True, "numero_areas": int(ap["nombre"].nunique()),
        "protegida_ha": rd(ap_ha, 1), "protegida_pct": rd(ap_ha / AREA_T_HA * 100, 1),
        "por_categoria": _agr("categoria"), "por_ambito": _agr("ambito"), "por_naturaleza": _agr("naturaleza"),
        "areas": por_area,
    }
    print(f"{K['areas_protegidas']['numero_areas']} área(s) protegida(s) · {fmt(ap_ha,1)} ha ({fmt(ap_ha/AREA_T_HA*100,1)} % del territorio)")
else:
    K["areas_protegidas"] = {"hay_traslape": False, "numero_areas": 0, "protegida_ha": 0.0, "protegida_pct": 0.0,
                             "por_categoria": [], "por_ambito": [], "por_naturaleza": [], "areas": []}
    print("Sin áreas protegidas del RUNAP que se traslapen con el territorio.")

In [ ]:
#@title 8 · KPI · Hidrografía { display-mode: "form" }
def _km(g):
    return float(g.length.sum()) / 1000 if len(g) else 0.0

dr, rp = D["drenajes"].copy(), D["rios_principales"].copy()
km_dr, km_rp = _km(dr), _km(rp)

por_tipo = []
if len(dr):
    c_t = col(dr, "LAYER")
    dr["tipo"] = texto_col(dr, "LAYER") if c_t else "s/d"
    for k, g in dr.groupby("tipo"):
        por_tipo.append({"codigo": k, "km": rd(_km(g), 2), "tramos": int(len(g))})
    por_tipo.sort(key=lambda d: -(d["km"] or 0))

def _top_nombres(gdf, campo, n=10):
    if len(gdf) == 0 or col(gdf, campo) is None:
        return []
    g = gdf.copy()
    g["_n"] = g[col(g, campo)].astype(str).str.strip()
    g = g[(g["_n"] != "") & (~g["_n"].str.lower().isin(["none", "nan", "null"]))]
    out = [{"nombre": titulo(k), "km": rd(_km(x), 2)} for k, x in g.groupby("_n")]
    return sorted(out, key=lambda d: -(d["km"] or 0))[:n]

agua_ha = 0.0
if CLC_OK:
    agua_ha = float(clc[clc.n1 == "5"]["ha"].sum())
K["hidrografia"] = {
    "drenajes_km": rd(km_dr, 2), "drenajes_tramos": int(len(dr)),
    "densidad_drenaje_km_km2": rd(km_dr / AREA_T_KM2, 3) if AREA_T_KM2 else None,
    "por_tipo": por_tipo, "drenajes_nombrados": _top_nombres(dr, "NMG"),
    "rios_principales_km": rd(km_rp, 2), "rios_principales": _top_nombres(rp, "Name"),
    "cuerpos_agua_clc_ha": rd(agua_ha, 1), "cuerpos_agua_clc_pct": rd(agua_ha / AREA_T_HA * 100, 2),
    "nota": "La capa de drenajes es la hidrografía disponible en ArcGIS Online (no es la red hídrica oficial IDEAM completa); los ríos principales de Interés Colombia son solo una selección.",
}
print(f"Drenajes: {fmt(km_dr,1)} km · densidad {fmt(km_dr/AREA_T_KM2,2)} km/km² · ríos principales: {fmt(km_rp,1)} km · agua (CLC): {fmt(agua_ha,1)} ha")

In [ ]:
#@title 9 · KPI · Vías y accesibilidad { display-mode: "form" }
por_clase, partes, km_tot = [], [], 0.0
for cid, etiqueta in VIAS.items():
    g = D[cid]
    km = _km(g)
    km_tot += km
    por_clase.append({"id": cid, "nombre": etiqueta, "km": rd(km, 2), "tramos": int(len(g))})
    if len(g):
        gg = g[["geometry"]].copy()
        c_nm = col(g, "NOMBRE_GEOGRAFICO")
        gg["nombre"] = g[c_nm].astype(str).str.strip() if c_nm else ""
        gg["clase"] = cid
        partes.append(gg)
for d in por_clase:
    d["porcentaje"] = rd((d["km"] or 0) / km_tot * 100, 1) if km_tot else 0.0

VIAS_ALL = pd.concat(partes, ignore_index=True) if partes else vacio(CRS_METRICO)
VIAS_GEOM = unary_union(VIAS_ALL.geometry) if len(VIAS_ALL) else None

# accesibilidad: qué parte del territorio queda a <= d km de una vía
acceso = []
BUFFERS = {}
for d in DISTANCIAS_ACCESO_KM:
    if VIAS_GEOM is not None:
        b = VIAS_GEOM.buffer(d * 1000).intersection(TERR_GEOM)
        BUFFERS[d] = b
        ha = b.area / 1e4
    else:
        BUFFERS[d], ha = None, 0.0
    acceso.append({"distancia_km": d, "ha": rd(ha, 1), "porcentaje": rd(ha / AREA_T_HA * 100, 1)})

nombradas = []
if len(VIAS_ALL):
    v = VIAS_ALL[~VIAS_ALL["nombre"].str.lower().isin(["", "none", "nan", "null"])]
    for k, g in v.groupby("nombre"):
        nombradas.append({"nombre": titulo(k), "km": rd(_km(g), 2)})
    nombradas.sort(key=lambda d: -(d["km"] or 0))

pav = next((d["km"] for d in por_clase if d["id"] == "vias_pavimentadas"), 0) or 0
K["vias"] = {
    "total_km": rd(km_tot, 2), "tramos": int(sum(d["tramos"] for d in por_clase)),
    "densidad_km_km2": rd(km_tot / AREA_T_KM2, 3) if AREA_T_KM2 else None,
    "pavimentadas_pct": rd(pav / km_tot * 100, 1) if km_tot else 0.0,
    "por_clase": por_clase, "accesibilidad": acceso, "vias_nombradas": nombradas[:8],
    "territorio_sin_via_3km_pct": rd(100 - (acceso[-1]["porcentaje"] or 0), 1) if acceso else None,
    "nota": "Las vías provienen de la capa Interés Colombia (IGAC); sirven para lectura territorial, no para ingeniería. La clasificación es la de cada capa (pavimentadas, sin pavimentar, transitables en tiempo seco).",
}
print(f"Vías: {fmt(km_tot,1)} km · densidad {fmt(km_tot/AREA_T_KM2,2)} km/km² · pavimentadas {fmt(K['vias']['pavimentadas_pct'],1)} %")
for a in acceso:
    print(f"  a ≤ {a['distancia_km']} km de una vía: {fmt(a['porcentaje'],1)} % del territorio")

In [ ]:
#@title 10 · KPI · Cruces entre capas y tabla por vereda { display-mode: "form" }
def inter(a, b):
    """Intersección tolerante a geometrías problemáticas."""
    if a is None or b is None or a.is_empty or b.is_empty:
        return None
    try:
        return a.intersection(b)
    except Exception:
        return shapely.make_valid(a).intersection(shapely.make_valid(b))

def km_de(g):
    return 0.0 if g is None or g.is_empty else g.length / 1000

def ha_de(g):
    return 0.0 if g is None or g.is_empty else g.area / 1e4

DR_GEOM = unary_union(D["drenajes"].geometry) if len(D["drenajes"]) else None
BOSQUE_GEOM = unary_union(bosque.geometry) if CLC_OK and len(bosque) else None
NAT_GEOM = unary_union(clc[clc.n1.isin(NATURALES)].geometry) if CLC_OK and (clc.n1.isin(NATURALES)).any() else None

cr = {}
cr["vias_en_areas_protegidas_km"] = rd(km_de(inter(VIAS_GEOM, AP_GEOM)), 2)
cr["drenajes_en_areas_protegidas_km"] = rd(km_de(inter(DR_GEOM, AP_GEOM)), 2)
cr["bosque_en_areas_protegidas_ha"] = rd(ha_de(inter(BOSQUE_GEOM, AP_GEOM)), 1)
bosque_ha_v = (K["cobertura"] or {}).get("bosque_ha") or 0
cr["bosque_protegido_pct"] = rd(cr["bosque_en_areas_protegidas_ha"] / bosque_ha_v * 100, 1) if bosque_ha_v else None
cr["natural_en_areas_protegidas_ha"] = rd(ha_de(inter(NAT_GEOM, AP_GEOM)), 1)
cr["drenajes_en_bosque_km"] = rd(km_de(inter(DR_GEOM, BOSQUE_GEOM)), 2)
cr["vias_en_bosque_km"] = rd(km_de(inter(VIAS_GEOM, BOSQUE_GEOM)), 2)
cerca = []
for d in DISTANCIAS_ACCESO_KM:
    ha = ha_de(inter(BOSQUE_GEOM, BUFFERS.get(d)))
    cerca.append({"distancia_km": d, "ha": rd(ha, 1), "porcentaje_del_bosque": rd(ha / bosque_ha_v * 100, 1) if bosque_ha_v else None})
cr["bosque_cerca_de_vias"] = cerca
cr["nota"] = "Cruces espaciales de lectura territorial: no son medidas de presión ni de amenaza por sí mismas."
K["cruces"] = cr
print("Cruces:", {k: v for k, v in cr.items() if k not in ("nota", "bosque_cerca_de_vias")})
for c_ in cerca:
    print(f"  bosque a ≤ {c_['distancia_km']} km de una vía: {fmt(c_['ha'],1)} ha ({fmt(c_['porcentaje_del_bosque'],1)} % del bosque)")

# ---------------- tabla por vereda ----------------
def tabla_veredas():
    if len(VER_M) == 0:
        return pd.DataFrame()
    v = VER_M[["cod_ver", "nom_ver", "geometry"]].copy()
    v["cod_ver"] = v["cod_ver"].astype(str)
    base = pd.DataFrame({"cod_ver": v["cod_ver"].values, "vereda": v["nom_ver"].values, "area_ha": v.geometry.area.values / 1e4})
    c_ao = col(VER_M, "area_ha")
    if c_ao:
        base["area_oficial_ha"] = pd.to_numeric(VER_M[c_ao], errors="coerce").values
    base = base.set_index("cod_ver")

    def sobre(layer, cols, tipo):
        if layer is None or len(layer) == 0:
            return None
        cols = [c for c in cols if c in layer.columns]
        try:
            o = gpd.overlay(layer[cols + ["geometry"]], v[["cod_ver", "geometry"]], how="intersection", keep_geom_type=True)
        except Exception:
            return None
        return limpiar(o, tipo)

    ser = lambda s: s.reindex(base.index).fillna(0.0) if s is not None else pd.Series(0.0, index=base.index)

    # coberturas
    if CLC_OK:
        o = sobre(clc, ["n1", "n2", "leyenda"], "poly")
        if o is not None and len(o):
            o["ha"] = o.geometry.area / 1e4
            tot = o.groupby("cod_ver")["ha"].sum()
            base["pct_natural"] = (ser(o[o.n1.isin(NATURALES)].groupby("cod_ver")["ha"].sum()) / ser(tot).replace(0, np.nan) * 100).round(1)
            base["bosque_ha"] = ser(o[o.n2 == "31"].groupby("cod_ver")["ha"].sum()).round(1)
            base["bosque_pct"] = (base["bosque_ha"] / ser(tot).replace(0, np.nan) * 100).round(1)
            g = o.groupby(["cod_ver", "leyenda"])["ha"].sum().reset_index().sort_values("ha", ascending=False).drop_duplicates("cod_ver").set_index("cod_ver")
            base["cobertura_dominante"] = g["leyenda"].reindex(base.index)
            base["dominante_pct"] = (g["ha"].reindex(base.index) / ser(tot).replace(0, np.nan) * 100).round(1)
    # áreas protegidas
    if AP_GEOM is not None:
        o = sobre(gpd.GeoDataFrame({"geometry": [AP_GEOM]}, crs=CRS_METRICO), [], "poly")
        s = o.assign(ha=o.geometry.area / 1e4).groupby("cod_ver")["ha"].sum() if o is not None and len(o) else None
    else:
        s = None
    base["protegida_ha"] = ser(s).round(1)
    base["protegida_pct"] = (base["protegida_ha"] / base["area_ha"] * 100).round(1)
    # vías y drenajes
    o = sobre(VIAS_ALL, ["clase"], "line") if len(VIAS_ALL) else None
    if o is not None and len(o):
        o["km"] = o.geometry.length / 1000
        base["vias_km"] = ser(o.groupby("cod_ver")["km"].sum()).round(2)
        base["vias_pavimentadas_km"] = ser(o[o.clase == "vias_pavimentadas"].groupby("cod_ver")["km"].sum()).round(2)
    else:
        base["vias_km"], base["vias_pavimentadas_km"] = 0.0, 0.0
    base["densidad_vias_km_km2"] = (base["vias_km"] / (base["area_ha"] / 100)).round(3)
    o = sobre(D["drenajes"], [], "line") if len(D["drenajes"]) else None
    base["drenajes_km"] = ser(o.assign(km=o.geometry.length / 1000).groupby("cod_ver")["km"].sum() if o is not None and len(o) else None).round(2)
    base["densidad_drenaje_km_km2"] = (base["drenajes_km"] / (base["area_ha"] / 100)).round(3)
    base["area_ha"] = base["area_ha"].round(1)
    return base.reset_index().sort_values("area_ha", ascending=False).reset_index(drop=True)

TAB_VEREDAS = tabla_veredas()
K["veredas"] = json.loads(TAB_VEREDAS.where(TAB_VEREDAS.notna(), None).to_json(orient="records", force_ascii=False)) if len(TAB_VEREDAS) else []
print(f"Tabla por vereda: {len(TAB_VEREDAS)} fila(s)")
if len(TAB_VEREDAS):
    print(TAB_VEREDAS.head(8).to_string(index=False))

In [ ]:
#@title 11 · Lecturas automáticas, tarjetas KPI y kpis.json { display-mode: "form" }
T, C, A, H, V, X = K["territorio"], K["cobertura"], K["areas_protegidas"], K["hidrografia"], K["vias"], K["cruces"]
fecha = datetime.datetime.now().strftime("%Y-%m-%d")

# ---------- lecturas en lenguaje claro ----------
L = []
L.append(f"{T['nombre']} ocupa {fmt(T['area_ha'],1)} ha ({fmt(T['area_km2'],2)} km²) en {T['departamento']}"
         + (f" y equivale al {fmt(T['porcentaje_del_municipio'],1)} % del municipio de {T['municipio']}." if T["tipo"] == "vereda" else
            (f"; tiene {T['numero_veredas_municipio']} veredas registradas." if T.get("numero_veredas_municipio") else ".")))
if C:
    L.append(f"La cobertura dominante es «{C['cobertura_dominante']['nombre']}» ({fmt(C['cobertura_dominante']['porcentaje'],1)} % del área clasificada); "
             f"el {fmt(C['natural_seminatural_pct'],1)} % del territorio conserva coberturas naturales o seminaturales y el {fmt(C['transformada_pct'],1)} % está transformado (territorios artificializados y agrícolas).")
    if (C["bosque_ha"] or 0) > 0:
        f_ = C["bosque_fragmentacion"]
        L.append(f"Hay {fmt(C['bosque_ha'],1)} ha de bosque ({fmt(C['bosque_pct'],1)} %) repartidas en {f_['parches']} parche(s); el mayor reúne el {fmt(f_['porcentaje_en_parche_mayor'],1)} % del bosque.")
    L.append(f"Se identifican {C['numero_coberturas_nivel3']} coberturas de nivel 3; el índice de Shannon es {fmt(C['shannon'],2)} (equitatividad {fmt(C['equitatividad'],2)}).")
if A["hay_traslape"]:
    mayor = A["areas"][0]
    L.append(f"{A['numero_areas']} área(s) protegida(s) del RUNAP se traslapan con el territorio y cubren {fmt(A['protegida_ha'],1)} ha ({fmt(A['protegida_pct'],1)} %); la mayor es {mayor['nombre']} ({mayor['categoria']}).")
else:
    L.append("No se traslapa ninguna área protegida del RUNAP con el territorio.")
if (H["drenajes_km"] or 0) > 0:
    L.append(f"La capa de drenajes suma {fmt(H['drenajes_km'],1)} km, con una densidad de {fmt(H['densidad_drenaje_km_km2'],2)} km/km².")
else:
    L.append("La capa de drenajes consultada no registra cauces dentro del territorio.")
if (V["total_km"] or 0) > 0:
    ac = V["accesibilidad"]
    L.append(f"Hay {fmt(V['total_km'],1)} km de vías ({fmt(V['pavimentadas_pct'],1)} % pavimentadas) y el {fmt(ac[0]['porcentaje'],1)} % del territorio queda a ≤ {ac[0]['distancia_km']} km de una vía"
             + (f" ({fmt(ac[-1]['porcentaje'],1)} % a ≤ {ac[-1]['distancia_km']} km)." if len(ac) > 1 else "."))
else:
    L.append("La capa de vías consultada no registra vías dentro del territorio.")
if (X["vias_en_areas_protegidas_km"] or 0) > 0:
    L.append(f"{fmt(X['vias_en_areas_protegidas_km'],1)} km de vías atraviesan áreas protegidas.")
if X["bosque_cerca_de_vias"] and X["bosque_cerca_de_vias"][0]["porcentaje_del_bosque"] is not None:
    b0 = X["bosque_cerca_de_vias"][0]
    L.append(f"El {fmt(b0['porcentaje_del_bosque'],1)} % del bosque está a ≤ {b0['distancia_km']} km de una vía (accesibilidad al bosque).")
K["lecturas"] = L

# ---------- fuentes (formato de nota bajo gráficos y tablas) ----------
K["fuentes"] = {
    "limites": f"DANE (s. f.). Marco Geoestadístico Nacional: municipios y veredas [Servicio ArcGIS REST publicado por IDEAM]. https://visualizador.ideam.gov.co/",
    "cobertura": "IDEAM (2024). Mapa de coberturas de la tierra, metodología Corine Land Cover adaptada para Colombia, escala 1:100.000 [Servicio ArcGIS REST]. https://visualizador.ideam.gov.co/",
    "areas_protegidas": "Ministerio de Ambiente y Desarrollo Sostenible (s. f.). Registro Único Nacional de Áreas Protegidas (RUNAP) [Servicio ArcGIS REST]. https://geo.minambiente.gov.co/",
    "hidrografia": "Capa HIDROGRAFIA (ArcGIS Online) e Instituto Geográfico Agustín Codazzi (s. f.), Interés Colombia: ríos principales [Servicios ArcGIS REST].",
    "vias": "Instituto Geográfico Agustín Codazzi (s. f.). Interés Colombia: vías [Servicio ArcGIS REST, ArcGIS Online].",
    "procesamiento": f"Elaboración propia con GeoIA Colombia · TerraDrop. Áreas y longitudes calculadas en MAGNA-SIRGAS Origen Nacional ({CRS_METRICO}); consulta de los servicios y cálculo el {fecha}.",
}

# ---------- tarjetas listas para el dashboard ----------
def tarjeta(grupo, id_, titulo_, valor, unidad, detalle, fuente):
    texto = f"{fmt(valor, 1 if isinstance(valor, float) and abs(valor) < 1000 else 0)} {unidad}".strip() if num(valor) is not None else str(valor)
    return {"grupo": grupo, "id": id_, "titulo": titulo_, "valor": valor, "unidad": unidad, "valor_texto": texto, "detalle": detalle, "fuente": fuente}

TARJ = [
    tarjeta("territorio", "area", "Área del territorio", T["area_ha"], "ha", f"{fmt(T['area_km2'],2)} km² · perímetro {fmt(T['perimetro_km'],1)} km", "limites"),
]
if T["tipo"] == "vereda":
    TARJ.append(tarjeta("territorio", "peso_municipio", "Peso en el municipio", T["porcentaje_del_municipio"], "%", f"de {T['municipio']}", "limites"))
elif T.get("numero_veredas_municipio") is not None:
    TARJ.append(tarjeta("territorio", "veredas", "Veredas", T["numero_veredas_municipio"], "", "registradas en el municipio", "limites"))
if C:
    TARJ += [
        tarjeta("cobertura", "natural", "Cobertura natural y seminatural", C["natural_seminatural_pct"], "%", "bosques, vegetación, humedales y agua", "cobertura"),
        tarjeta("cobertura", "bosque", "Bosque", C["bosque_ha"], "ha", f"{fmt(C['bosque_pct'],1)} % del territorio · {C['bosque_fragmentacion']['parches']} parche(s)", "cobertura"),
        tarjeta("cobertura", "dominante", "Cobertura dominante", C["cobertura_dominante"]["nombre"], "", f"{fmt(C['cobertura_dominante']['porcentaje'],1)} % del área", "cobertura"),
        tarjeta("cobertura", "shannon", "Diversidad de coberturas", C["shannon"], "H′", f"{C['numero_coberturas_nivel3']} coberturas de nivel 3", "cobertura"),
    ]
TARJ += [
    tarjeta("areas_protegidas", "protegida_pct", "Territorio en áreas protegidas", A["protegida_pct"], "%", f"{fmt(A['protegida_ha'],1)} ha · {A['numero_areas']} área(s)", "areas_protegidas"),
    tarjeta("hidrografia", "drenajes_km", "Drenajes", H["drenajes_km"], "km", f"{H['drenajes_tramos']} tramos", "hidrografia"),
    tarjeta("hidrografia", "densidad", "Densidad de drenaje", H["densidad_drenaje_km_km2"], "km/km²", "longitud de drenajes por km²", "hidrografia"),
    tarjeta("vias", "total_km", "Vías", V["total_km"], "km", f"{fmt(V['densidad_km_km2'],2)} km/km²", "vias"),
    tarjeta("vias", "pavimentadas", "Vías pavimentadas", V["pavimentadas_pct"], "%", "del total de vías", "vias"),
]
for a_ in V["accesibilidad"]:
    TARJ.append(tarjeta("vias", f"acceso_{a_['distancia_km']}km", f"Territorio a ≤ {a_['distancia_km']} km de una vía", a_["porcentaje"], "%", f"{fmt(a_['ha'],1)} ha", "vias"))
TARJ.append(tarjeta("cruces", "vias_en_ap", "Vías dentro de áreas protegidas", X["vias_en_areas_protegidas_km"], "km", "cruce de vías × RUNAP", "areas_protegidas"))
if X["bosque_cerca_de_vias"]:
    b0 = X["bosque_cerca_de_vias"][0]
    TARJ.append(tarjeta("cruces", "bosque_cerca_vias", f"Bosque a ≤ {b0['distancia_km']} km de una vía", b0["porcentaje_del_bosque"], "%", f"{fmt(b0['ha'],1)} ha", "cobertura"))
K["tarjetas"] = TARJ

K["metadatos"] = {
    "titulo": f"Tablero territorial · {T['nombre']}", "generado": datetime.datetime.now().isoformat(timespec="seconds"),
    "generador": "GeoIA Colombia · TerraDrop · Colab KPI", "origen_datos": FUENTE.get("origen"), "crs_metrico": CRS_METRICO,
    "capas": {c: {"nombre": SERV[c]["nombre"], "fuente": SERV[c]["fuente"], "elementos_en_territorio": int(len(D[c]))} for c in CAPAS},
    "grupos_kpi": ["territorio", "cobertura", "areas_protegidas", "hidrografia", "vias", "cruces"],
}
print("\n".join("• " + l for l in L))
print(f"\n{len(TARJ)} tarjetas KPI · {len(K['veredas'])} veredas en la tabla")

In [ ]:
#@title 12 · Exportar kpis.json, tablas CSV y GeoJSON livianos { display-mode: "form" }
OUT = os.path.join(CARPETA_SALIDA, f"{COD_VER or COD_MUN}_{slug(NOM_TERR)}")
for sub in ("", "geojson", "tablas"):
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

def redondear(o, n=5):
    if isinstance(o, float):
        return round(o, n)
    if isinstance(o, list):
        return [redondear(x, n) for x in o]
    if isinstance(o, dict):
        return {k: redondear(v, n) for k, v in o.items()}
    return o

def ligero(gdf, columnas, tol_m, max_kb=1200):
    """GeoJSON en WGS84, simplificado y con 5 decimales; sube la tolerancia hasta caber en max_kb."""
    if gdf is None or len(gdf) == 0:
        return {"type": "FeatureCollection", "features": []}
    cols = [c for c in columnas if c in gdf.columns]
    for _ in range(7):
        g = gdf[cols + ["geometry"]].copy()
        g["geometry"] = g.geometry.simplify(tol_m, preserve_topology=True)
        g = g[~g.geometry.is_empty].to_crs(4326)
        fc = redondear(json.loads(g.to_json(drop_id=True)))
        if len(json.dumps(fc, ensure_ascii=False)) / 1024 <= max_kb:
            break
        tol_m *= 1.8
    return fc

x0, y0, x1, y1 = TERR_GEOM.bounds
TOL = max(3.0, math.hypot(x1 - x0, y1 - y0) / 1500)   # metros

capas_geo = {}
t = TERR.dissolve().reset_index(drop=True)
t["nombre"], t["tipo"], t["area_ha"] = NOM_TERR_TITULO, K["territorio"]["tipo"], K["territorio"]["area_ha"]
capas_geo["territorio"] = ligero(t, ["nombre", "tipo", "area_ha"], TOL / 2)
if ES_VEREDA:
    m_ = mun.copy(); m_["nombre"] = NOM_MUN
    capas_geo["municipio"] = ligero(m_, ["nombre"], TOL)
if len(VER_M):
    vv = VER_M.copy(); vv["cod_ver"] = vv["cod_ver"].astype(str)
    if len(TAB_VEREDAS):
        vv = vv.merge(TAB_VEREDAS.drop(columns=["vereda"]), on="cod_ver", how="left")
    capas_geo["veredas"] = ligero(vv, [c for c in vv.columns if c != "geometry"], TOL)
if CLC_OK:
    cc = clc.dissolve(by="codigo", aggfunc={"leyenda": "first", "n1": "first", "ha": "sum"}).reset_index()
    cc["color"] = cc["n1"].map(CLC_COL); cc["ha"] = cc["ha"].round(1)
    capas_geo["coberturas"] = ligero(cc, ["codigo", "leyenda", "n1", "color", "ha"], TOL)
if AP_GEOM is not None:
    capas_geo["areas_protegidas"] = ligero(ap, ["nombre", "categoria", "naturaleza", "ambito", "estado"], TOL)
if len(VIAS_ALL):
    va = VIAS_ALL.copy(); va["clase_nombre"] = va["clase"].map(VIAS)
    capas_geo["vias"] = ligero(va, ["clase", "clase_nombre", "nombre"], TOL / 2)
if len(D["drenajes"]):
    d_ = D["drenajes"].copy()
    capas_geo["drenajes"] = ligero(d_, ["tipo", "NMG"], TOL / 2)
if len(D["rios_principales"]):
    capas_geo["rios_principales"] = ligero(D["rios_principales"], ["Name"], TOL / 2)

ARCH_GEO = {}
for nombre, fc in capas_geo.items():
    ruta_ = os.path.join(OUT, "geojson", f"{nombre}.geojson")
    with open(ruta_, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False, separators=(",", ":"))
    ARCH_GEO[nombre] = {"archivo": f"geojson/{nombre}.geojson", "elementos": len(fc["features"]), "kb": round(os.path.getsize(ruta_) / 1024)}
K["archivos_geojson"] = ARCH_GEO

# ---- tablas CSV ----
def csv(nombre, df):
    if df is not None and len(df):
        df.to_csv(os.path.join(OUT, "tablas", nombre), index=False, encoding="utf-8-sig")

if C:
    csv("cobertura_nivel1.csv", pd.DataFrame(C["nivel1"]).drop(columns=["color"]))
    csv("cobertura_nivel2.csv", pd.DataFrame(C["nivel2"]))
    csv("cobertura_nivel3.csv", pd.DataFrame(C["nivel3"]))
csv("areas_protegidas.csv", pd.DataFrame(A["areas"]))
csv("hidrografia_tipos.csv", pd.DataFrame(H["por_tipo"]))
csv("hidrografia_nombrados.csv", pd.DataFrame(H["drenajes_nombrados"]))
csv("rios_principales.csv", pd.DataFrame(H["rios_principales"]))
csv("vias_clases.csv", pd.DataFrame(V["por_clase"]))
csv("accesibilidad.csv", pd.DataFrame(V["accesibilidad"]))
csv("veredas.csv", TAB_VEREDAS)
csv("tarjetas_kpi.csv", pd.DataFrame(K["tarjetas"]))

with open(os.path.join(OUT, "kpis.json"), "w", encoding="utf-8") as f:
    json.dump(K, f, ensure_ascii=False, indent=2, default=str)

print("Carpeta de salida:", OUT)
for n, i in ARCH_GEO.items():
    print(f"  geojson/{n}.geojson · {i['elementos']} elementos · {i['kb']} kB")
print("  kpis.json ·", round(os.path.getsize(os.path.join(OUT, 'kpis.json')) / 1024), "kB")

In [ ]:
#@title 13 · Mapa de ubicación (PNG) { display-mode: "form" }
from matplotlib import pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

FONDO, TINTA, NARANJA, PAPEL = "#f6f1e7", "#0d0d12", "#ff9f1c", "#fffaf0"

def contexto_departamentos():
    """Departamentos simplificados para los recuadros de ubicación. Si no hay red, devuelve (None, None)."""
    try:
        todos = consultar("dep", url=URL_DEPARTAMENTOS, page=100, spatial=False,
                          extra={"maxAllowableOffset": 0.02, "geometryPrecision": 3})
        if len(todos) == 0:
            return None, None
        cc = col(todos, "cod_dane")
        clave = todos[cc].astype(str).str.replace(r"\.0$", "", regex=True).map(lambda s: s.lstrip("0") or "0")
        mio = todos[clave == str(int(COD_MUN[:2]))]
        det = None
        if len(mio):
            valor = mio[cc].iloc[0]
            cond = f"{cc}={valor}" if isinstance(valor, (int, float, np.integer)) else f"{cc}='{valor}'"
            det = consultar("dep", where=cond, url=URL_DEPARTAMENTOS, page=100, spatial=False,
                            extra={"maxAllowableOffset": 0.004, "geometryPrecision": 4})
            det = det if len(det) else mio
        return todos.assign(_clave=clave).to_crs(CRS_METRICO), (det.to_crs(CRS_METRICO) if det is not None else None)
    except Exception as e:
        print("  (sin contexto departamental:", str(e)[:80], ")")
        return None, None

def barra_escala(ax):
    xa, xb = ax.get_xlim(); ya, yb = ax.get_ylim()
    ancho = xb - xa
    opciones = [50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]
    L_ = max([o for o in opciones if o <= ancho / 3.5] or [opciones[0]])
    x_ini, y_ini, alto = xa + ancho * 0.04, ya + (yb - ya) * 0.045, (yb - ya) * 0.012
    for i in range(4):
        ax.add_patch(Rectangle((x_ini + i * L_ / 4, y_ini), L_ / 4, alto, fc=TINTA if i % 2 == 0 else "white", ec=TINTA, lw=.8, zorder=30))
    et = f"{L_/1000:g} km" if L_ >= 1000 else f"{L_} m"
    caja = dict(boxstyle="round,pad=.15", fc=PAPEL, ec="none", alpha=.9)
    ax.text(x_ini, y_ini + alto * 2.2, "0", fontsize=8, va="bottom", ha="center", zorder=30, bbox=caja)
    ax.text(x_ini + L_, y_ini + alto * 2.2, et, fontsize=8, va="bottom", ha="center", zorder=30, bbox=caja)

def mapa_ubicacion(ruta_png):
    deps, dep_sel = contexto_departamentos()
    fig = plt.figure(figsize=(16, 9), dpi=150, facecolor=FONDO)
    gs = GridSpec(2, 2, width_ratios=[0.33, 0.67], height_ratios=[1, 1], left=.03, right=.985, top=.855, bottom=.115, wspace=.035, hspace=.14)
    ax_c, ax_d, ax = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[:, 1])
    for a in (ax_c, ax_d, ax):
        a.set_facecolor(PAPEL); a.set_xticks([]); a.set_yticks([]); a.set_xlabel(""); a.set_ylabel("")
        for s in a.spines.values():
            s.set_color(TINTA); s.set_linewidth(1.2)
    pt = gpd.GeoSeries([TERR_GEOM.centroid], crs=CRS_METRICO)

    # --- Colombia ---
    ax_c.set_title("Colombia", fontsize=12, fontweight="bold", color=TINTA, loc="left")
    if deps is not None:
        deps.plot(ax=ax_c, color="#ddd5c3", edgecolor="white", linewidth=.5)
        sel = deps[deps["_clave"] == str(int(COD_MUN[:2]))]
        if len(sel):
            sel.plot(ax=ax_c, color=NARANJA, edgecolor=TINTA, linewidth=.8)
        pt.plot(ax=ax_c, color="#d92d20", markersize=26, edgecolor="white", linewidth=.8, zorder=5)
        ax_c.set_aspect("equal", adjustable="datalim")
    else:
        ax_c.text(.5, .5, "Contexto nacional\nno disponible\n(sin conexión)", ha="center", va="center", fontsize=10, color="#666", transform=ax_c.transAxes)

    # --- Departamento ---
    ax_d.set_title(f"Departamento de {NOM_DPTO}" if NOM_DPTO else "Departamento", fontsize=12, fontweight="bold", color=TINTA, loc="left")
    if dep_sel is not None and len(dep_sel):
        dep_sel.plot(ax=ax_d, color="#efe8d6", edgecolor=TINTA, linewidth=1)
        mun.plot(ax=ax_d, color=NARANJA, edgecolor=TINTA, linewidth=.8, alpha=.9 if not ES_VEREDA else .55)
        if ES_VEREDA:
            TERR.plot(ax=ax_d, color="#d92d20", edgecolor="#d92d20", linewidth=1.2)
        ax_d.set_aspect("equal", adjustable="datalim")
    else:
        mun.plot(ax=ax_d, color=NARANJA, edgecolor=TINTA, linewidth=.8)
        ax_d.set_aspect("equal", adjustable="datalim")
    mb = MUN_GEOM.centroid
    ax_d.annotate(NOM_MUN, (mb.x, mb.y), xytext=(0, 14), textcoords="offset points", ha="center", fontsize=9, fontweight="bold",
                  color=TINTA, bbox=dict(boxstyle="round,pad=.2", fc="white", ec=TINTA, lw=.6), zorder=9)

    # --- Territorio con capas ---
    ax.set_title("Territorio de estudio y capas de análisis", fontsize=12, fontweight="bold", color=TINTA, loc="left")
    if ES_VEREDA:
        mun.plot(ax=ax, color="#ece5d4", edgecolor="#9b927e", linewidth=.8, zorder=1)
    if CLC_OK and len(clc):
        clc.plot(ax=ax, color=clc["n1"].map(CLC_COL).fillna("#cccccc").tolist(), edgecolor="none", alpha=.82, zorder=2)
    else:
        TERR.plot(ax=ax, color="#ece5d4", edgecolor="none", zorder=2)
    if len(VER_M) > 1:
        VER_M.boundary.plot(ax=ax, color="#ffffff", linewidth=.5, alpha=.8, zorder=3)
    if AP_GEOM is not None:
        ap.plot(ax=ax, facecolor="none", edgecolor="#0a7f55", hatch="///", linewidth=1.4, zorder=4)
    if len(D["drenajes"]):
        D["drenajes"].plot(ax=ax, color="#2f9bff", linewidth=.6, zorder=5)
    if len(D["rios_principales"]):
        D["rios_principales"].plot(ax=ax, color="#0b4fcf", linewidth=1.8, zorder=6)
    for cid, color, lw, ls in (("vias_tiempo_seco", "#b9763f", 1.0, (0, (4, 2))), ("vias_sin_pavimentar", "#ff7ab0", 1.3, "-"), ("vias_pavimentadas", "#e53935", 2.0, "-")):
        if len(D[cid]):
            D[cid].plot(ax=ax, color=color, linewidth=lw, linestyle=ls, zorder=7)
    TERR.boundary.plot(ax=ax, color=TINTA, linewidth=2.6, zorder=10)
    mx, my = (x1 - x0) * .06, (y1 - y0) * .06
    ax.set_xlim(x0 - mx, x1 + mx); ax.set_ylim(y0 - my, y1 + my); ax.set_aspect("equal", adjustable="datalim")
    ax.apply_aspect()
    barra_escala(ax)
    ax.annotate("N", xy=(.955, .95), xytext=(.955, .83), xycoords="axes fraction", ha="center", fontsize=15, fontweight="bold", color=TINTA,
                arrowprops=dict(arrowstyle="-|>", color=TINTA, lw=2.2), zorder=40)

    for a in (ax_c, ax_d, ax):
        a.set_xlabel(""); a.set_ylabel("")
    # --- leyenda en una franja inferior ---
    items = [Patch(fc=CLC_COL[k], ec="none", alpha=.82, label=CLC_N1[k]) for k in CLC_N1] if CLC_OK else []
    items += [Patch(fc="none", ec="#0a7f55", hatch="///", label="Áreas protegidas (RUNAP)"),
              Line2D([], [], color="#2f9bff", lw=1, label="Drenajes"), Line2D([], [], color="#0b4fcf", lw=2.2, label="Ríos principales"),
              Line2D([], [], color="#e53935", lw=2.4, label="Vías pavimentadas"), Line2D([], [], color="#ff7ab0", lw=1.6, label="Vías sin pavimentar"),
              Line2D([], [], color="#b9763f", lw=1.2, ls=(0, (4, 2)), label="Vías en tiempo seco"), Line2D([], [], color=TINTA, lw=2.6, label="Límite del territorio")]
    fig.legend(handles=items, loc="lower center", ncol=6, fontsize=8.6, frameon=True, facecolor=PAPEL, edgecolor=TINTA, bbox_to_anchor=(.5, .028), columnspacing=1.2, handlelength=1.8)

    fig.text(.03, .945, f"Mapa de ubicación · {NOM_TERR_TITULO}", fontsize=24, fontweight="bold", color=TINTA, va="center")
    sub = f"{NOM_DPTO}, Colombia  ·  {fmt(AREA_T_HA, 1)} ha  ·  código DANE {COD_VER or COD_MUN}"
    fig.text(.03, .897, sub, fontsize=12.5, color="#4a4a55", va="center")
    fig.text(.03, .012, f"Fuentes: DANE (límites), IDEAM (Corine Land Cover 2024), MinAmbiente (RUNAP), IGAC (vías e hidrografía). Sistema de referencia: MAGNA-SIRGAS Origen Nacional ({CRS_METRICO}). "
             f"Elaboración propia con GeoIA Colombia · TerraDrop, {fecha}.", fontsize=7.6, color="#555", va="center")
    fig.savefig(ruta_png, dpi=150, facecolor=FONDO)
    plt.show()
    plt.close(fig)

RUTA_MAPA = os.path.join(OUT, "mapa_ubicacion.png")
mapa_ubicacion(RUTA_MAPA)
print("Mapa guardado:", RUTA_MAPA)

In [ ]:
#@title 14 · Generar el prompt de Lovable { display-mode: "form" }
def _lista_valores():
    return "\n".join(f"- **{t['titulo']}**: {t['valor_texto']}" + (f" — {t['detalle']}" if t.get("detalle") else "") for t in K["tarjetas"])

def _archivos():
    f = [f"- `kpis.json` → guárdalo en `public/data/kpis.json` (toda la información numérica y textual del tablero)."]
    desc = {"territorio": "límite del territorio de estudio", "municipio": "límite municipal (contexto)",
            "veredas": "veredas con sus indicadores (cod_ver, nom_ver, area_ha, pct_natural, bosque_ha, protegida_pct, vias_km, densidad_vias_km_km2, drenajes_km…)",
            "coberturas": "coberturas CLC 2024 por tipo (campos codigo, leyenda, n1, color, ha)",
            "areas_protegidas": "áreas protegidas RUNAP (nombre, categoria, ambito, naturaleza)", "vias": "vías (campo clase y clase_nombre)",
            "drenajes": "drenajes", "rios_principales": "ríos principales"}
    for n, i in ARCH_GEO.items():
        f.append(f"- `{i['archivo']}` → `public/data/{n}.geojson` · {desc.get(n, n)} ({i['elementos']} elementos, {i['kb']} kB).")
    f.append("- `mapa_ubicacion.png` → `public/img/mapa_ubicacion.png` (mapa de ubicación para la portada).")
    return "\n".join(f)

def prompt_completo():
    T_ = K["territorio"]
    tipo_txt = f"la vereda {T_['vereda']} del municipio de {T_['municipio']}" if T_["tipo"] == "vereda" else f"el municipio de {T_['municipio']}"
    veredas_txt = "Como el territorio es una vereda, la tabla se presenta como **ficha** (una sola fila) con los mismos indicadores." if ES_VEREDA else f"Hay {len(K['veredas'])} veredas en la tabla."
    return f"""# Prompt para Lovable · Tablero territorial «{T_['nombre']}»

Crea una aplicación web llamada **«Tablero territorial · {T_['nombre']}»** con React + Vite + TypeScript + Tailwind + shadcn/ui, gráficos con **Recharts** y mapa con **Leaflet (react-leaflet)**. Es un dashboard de **solo lectura**, 100 % estático (sin backend ni inicio de sesión), todo en **español de Colombia**, para sustentar el estudio territorial de {tipo_txt}, departamento de {T_['departamento']}. Los datos ya vienen procesados con SIG: **no inventes, no estimes y no recalcules ninguna cifra**.

## 1. Datos que voy a adjuntar
{_archivos()}

Reglas de datos:
- Toda cifra, etiqueta y texto analítico sale de `kpis.json` (carga con `fetch`). Las tarjetas están en `tarjetas`, los textos en `lecturas`, las fuentes en `fuentes`, las tablas en `cobertura`, `areas_protegidas`, `hidrografia`, `vias`, `cruces` y `veredas`.
- Si un bloque llega vacío o en `null`, muestra un estado vacío amable (p. ej. «No hay áreas protegidas del RUNAP que se traslapen con este territorio») en lugar de ocultar la sección.
- Formato colombiano: miles con punto y decimales con coma (1.234,5); áreas en **ha** y longitudes en **km**; porcentajes con un decimal.

## 2. Estructura de la página (una sola página, menú fijo con anclas y barra de progreso)
1. **Portada**: título grande, chips (departamento, municipio, vereda si aplica, área en ha, código DANE) y el `mapa_ubicacion.png` como imagen principal con botón «Descargar mapa». Debajo, «Lectura rápida» con las 3 primeras frases de `lecturas`.
2. **Resumen**: rejilla de tarjetas KPI con las tarjetas de los grupos `territorio`, `cobertura` y `areas_protegidas` (icono lucide, valor grande con animación de conteo, detalle pequeño).
3. **Territorio y cobertura**: dona con `cobertura.nivel1` (usa el campo `color`), barras horizontales con las 8 coberturas de nivel 3 más grandes, medidor de «cobertura natural y seminatural», bloque de fragmentación del bosque (parches, tamaño medio, parche mayor) y tooltip que explique el índice de Shannon en lenguaje sencillo.
4. **Áreas protegidas**: radial con `protegida_pct`, lista de `areas` (nombre, categoría, ámbito, ha en el territorio, % del territorio) con chips distintos para público/privado, y barras por categoría.
5. **Hidrografía**: tarjetas (km de drenajes, densidad km/km², ríos principales, cuerpos de agua CLC), barras con `drenajes_nombrados` y `rios_principales`, y un aviso discreto con `hidrografia.nota`.
6. **Vías y accesibilidad**: barras apiladas por clase de vía (km), dos semicírculos con el % del territorio a ≤ 1 km y ≤ 3 km de una vía, y ranking de `vias_nombradas`. Aviso con `vias.nota`.
7. **Cruces entre capas**: tarjetas con vías dentro de áreas protegidas, bosque protegido, drenajes en bosque y bosque cerca de vías, con la advertencia de `cruces.nota`.
8. **Mapa interactivo** (ver sección 4).
9. **Tabla por vereda**: buscador, columnas ordenables, barras de datos en `pct_natural`, `protegida_pct` y `densidad_vias_km_km2`, botón «Descargar CSV» y selector de indicador para colorear el mapa. Al pulsar una fila, el mapa hace zoom y resalta esa vereda. {veredas_txt}
10. **Lo que dicen los datos**: lista de `lecturas` como tarjetas cortas.
11. **Modo Aula** (pestaña o interruptor): 6 preguntas guía para discutir el territorio con estudiantes (cobertura dominante, qué tan transformado está, relación entre vías y bosque, qué protege el RUNAP aquí, qué vereda destaca y por qué, qué límites tienen los datos), cada una con un botón «ver pista» que lleve a la sección correspondiente.
12. **Fuentes y metodología** (pie de página): lista `fuentes`, el sistema de referencia ({CRS_METRICO}), la fecha de generación (`metadatos.generado`) y las limitaciones (escala 1:100.000 de CLC; capas de vías e hidrografía de Interés Colombia son de lectura territorial, no de ingeniería).

## 3. Valores de referencia (para que verifiques que el resultado coincide)
{_lista_valores()}

## 4. Mapa interactivo
- Base clara (CARTO Positron u OpenStreetMap), ajustado a los límites de `territorio.geojson`; contorno del territorio grueso y oscuro (#0d0d12).
- Capas con interruptores y leyenda plegable: coberturas (color del campo `color`), áreas protegidas (borde verde #0a7f55 con trama), drenajes (#2f9bff), ríos principales (#0b4fcf), vías (pavimentadas #e53935, sin pavimentar #ff7ab0, tiempo seco #b9763f en línea discontinua) y veredas (coropleta según el indicador elegido).
- Popups con los atributos de cada elemento (nombre, categoría, ha o km). Control de pantalla completa y botón «Volver al territorio».

## 5. Estilo visual «Clay»
- Fondo crema #f6f1e7, texto tinta #0d0d12, acentos naranja #ff9f1c, verde #27c98f, azul #2f9bff, rosa #ff7ab0 y morado #8b6cf0.
- Tarjetas tipo arcilla: radios de 28 px, sombra suave doble (luz arriba-izquierda, sombra abajo-derecha) y relieve sutil; botones redondeados con micro-animación al pasar el cursor.
- Tipografía geométrica redondeada (Sora o Nunito, de Google Fonts); títulos grandes y jerarquía clara.
- Diseño responsive (móvil primero), contraste AA, no depender solo del color (etiquetas e iconos), `aria-label` en gráficos y tablas con encabezados.

## 6. Fuentes bajo cada gráfico y tabla
Bajo **cada** gráfico, mapa o tabla añade una línea pequeña «Fuente: …» con el texto de `kpis.json → fuentes` que corresponda: `cobertura` (sección 3), `areas_protegidas` (4), `hidrografia` (5), `vias` (6), `limites` (tabla de veredas y mapa) y, en todos, `procesamiento`.

## 7. Criterios de aceptación
- Carga sin errores de consola y todas las cifras coinciden con la sección 3.
- Se ve bien en móvil y escritorio; el menú fijo resalta la sección activa.
- Las capas del mapa se activan y desactivan sin recargar; la tabla ordena y filtra.
- Ninguna sección muestra datos inventados: lo que no esté en los archivos aparece como estado vacío.
"""

def prompt_corto():
    T_ = K["territorio"]
    return f"""Crea un dashboard de solo lectura, en español de Colombia, llamado «Tablero territorial · {T_['nombre']}» ({T_['departamento']}) con React + Vite + TypeScript + Tailwind + shadcn/ui, Recharts y react-leaflet. Los datos ya están calculados: léelos de `public/data/kpis.json` y de los GeoJSON de `public/data/` (territorio, coberturas, areas_protegidas, vias, drenajes, rios_principales, veredas); no inventes ni recalcules cifras.

Secciones, en una sola página con menú fijo: 1) portada con `public/img/mapa_ubicacion.png` y chips (departamento, municipio, área en ha); 2) resumen con las tarjetas de `kpis.json → tarjetas`; 3) territorio y cobertura (dona nivel 1, barras nivel 3); 4) áreas protegidas; 5) hidrografía; 6) vías y accesibilidad (% del territorio a ≤ 1 y ≤ 3 km de una vía); 7) mapa interactivo con interruptores de capas; 8) tabla por vereda ordenable; 9) fuentes y metodología.

Estilo «Clay»: fondo crema #f6f1e7, tarjetas redondeadas con relieve suave, acentos naranja #ff9f1c, verde #27c98f y azul #2f9bff, tipografía redondeada. Formato colombiano de números (1.234,5), ha y km. Bajo cada gráfico y tabla pon una línea «Fuente: …» tomada de `kpis.json → fuentes`. Si un bloque viene vacío, muestra un estado vacío amable.

Haz primero esta estructura con datos reales; después te pediré el detalle de cada sección.
"""

with open(os.path.join(OUT, "prompt_lovable.md"), "w", encoding="utf-8") as f:
    f.write(prompt_completo())
with open(os.path.join(OUT, "prompt_lovable_corto.md"), "w", encoding="utf-8") as f:
    f.write(prompt_corto())
print(f"prompt_lovable.md ({len(prompt_completo())} caracteres) y prompt_lovable_corto.md ({len(prompt_corto())} caracteres) listos.\n")
print(prompt_corto())

In [ ]:
#@title 15 · Empaquetar y descargar { display-mode: "form" }
LEEME = f"""GeoIA Colombia · TerraDrop → Dashboard en Lovable
Territorio: {NOM_TERR_TITULO} ({NOM_DPTO}) · generado {fecha}

CONTENIDO
- prompt_lovable_corto.md : primer mensaje para Lovable (estructura general).
- prompt_lovable.md       : prompt completo con secciones, estilo, mapa y criterios de aceptación.
- kpis.json               : indicadores (tarjetas, coberturas, áreas protegidas, hidrografía, vías, cruces, veredas, lecturas, fuentes).
- geojson/                : territorio y capas livianas para el mapa.
- tablas/                 : tablas CSV de cada grupo de KPI.
- mapa_ubicacion.png      : mapa de ubicación para la portada.

CÓMO USARLO EN LOVABLE
1. Crea un proyecto nuevo en https://lovable.dev y pega prompt_lovable_corto.md.
2. Arrastra al chat kpis.json, la carpeta geojson/ y mapa_ubicacion.png (o súbelos a public/data y public/img).
3. Cuando la estructura esté lista, pega prompt_lovable.md para afinar cada sección.
4. Revisa que las cifras coincidan con la sección «Valores de referencia» del prompt.

NOTAS
- Áreas y longitudes calculadas en {CRS_METRICO}. Las capas son de lectura territorial, no de ingeniería ni de linderos legales.
- Revisa la vigencia de cada fuente en la página «Fuentes y calidad» de TerraDrop.
"""
with open(os.path.join(OUT, "LEEME.txt"), "w", encoding="utf-8") as f:
    f.write(LEEME)

ZIP_SALIDA = os.path.join(CARPETA_SALIDA, f"paquete_lovable_{COD_VER or COD_MUN}_{slug(NOM_TERR)}.zip")
with zipfile.ZipFile(ZIP_SALIDA, "w", zipfile.ZIP_DEFLATED) as z:
    for raiz, _, archivos in os.walk(OUT):
        for a in archivos:
            p = os.path.join(raiz, a)
            z.write(p, os.path.relpath(p, OUT))
print(f"Paquete listo: {ZIP_SALIDA} ({os.path.getsize(ZIP_SALIDA) / 1024:.0f} kB)")
for raiz, _, archivos in os.walk(OUT):
    for a in sorted(archivos):
        print("  ", os.path.relpath(os.path.join(raiz, a), OUT))
try:
    from google.colab import files
    files.download(ZIP_SALIDA)
except ImportError:
    print("\n(Fuera de Colab: el ZIP quedó en la carpeta de salida.)")